# Movie Recommendation System

## Phase 1: Content-Based Filtering

### Project Overview

The goal of this project is to build a Movie Recommendation System that suggests movies to users based on the similarities between movies.

We are using the TMDB 5000 Movies dataset and the TMDB 5000 Credits dataset.

### What is Content-Based Filtering?

Content-Based Filtering is a recommendation technique that suggests items based on their features and characteristics.

In our project, we compare movies using information such as:

- Movie overview
- Genres
- Keywords
- Cast
- Directors

Movies with similar content will receive higher similarity scores and can be recommended to each other.

### Techniques Used

1. Data Cleaning and Preprocessing
2. Text Processing and Stemming
3. CountVectorizer
4. Cosine Similarity
5. Movie Recommendation Function
6. Saving and Loading using Pickle

### Project Roadmap

- Phase 1: Content-Based Filtering
- Phase 2: Collaborative Filtering
- Phase 3: Neural Recommendation System
- Phase 4: Hybrid Recommendation System
- Phase 5: Evaluation and Deployment

This notebook focuses only on Phase 1.

## 2. Importing Libraries

Before working with the datasets, we need to import the Python libraries required for our project.

Libraries provide pre-written functions and tools that make it easier to perform tasks such as data manipulation, text processing, mathematical calculations and saving files.

We will use the following libraries:

- **Pandas:** For loading, cleaning and manipulating tabular data.
- **AST:** For converting string representations of Python objects into actual Python objects.
- **RE:** For working with text patterns and removing numeric tokens.
- **Pickle:** For saving and loading Python objects.
- **Pathlib:** For handling file and folder paths.
- **NLTK PorterStemmer:** For reducing words to their root-like forms.
- **CountVectorizer:** For converting text into numerical word-count vectors.
- **Cosine Similarity:** For measuring similarity between movie vectors.

In [1]:
# Importing libraries for data handling and preprocessing
import pandas as pd
import ast
import re
import pickle

# Importing Path for handling file and folder paths
from pathlib import Path

# Importing PorterStemmer for text stemming
from nltk.stem.porter import PorterStemmer

# Importing CountVectorizer for converting text into numerical vectors
from sklearn.feature_extraction.text import CountVectorizer

# Importing cosine_similarity for calculating similarity between movies
from sklearn.metrics.pairwise import cosine_similarity

## 3. Loading the Datasets

For this project, we are using two datasets from TMDB:

1. **TMDB 5000 Movies Dataset:** Contains movie information such as titles, genres, keywords, overviews, ratings and popularity.
2. **TMDB 5000 Credits Dataset:** Contains information about the cast and crew of each movie.

We will load both CSV files using Pandas' `read_csv()` function.

A DataFrame is a two-dimensional tabular data structure consisting of rows and columns.

In [3]:
# ============================================================
# LOADING THE DATASETS
# ============================================================

# Load the movies dataset
movies = pd.read_csv("../data/tmdb_5000_movies.csv")

# Load the credits dataset
credits = pd.read_csv("../data/tmdb_5000_credits.csv")

# Display dataset dimensions
print("Movies dataset shape:", movies.shape)
print("Credits dataset shape:", credits.shape)

Movies dataset shape: (4803, 20)
Credits dataset shape: (4803, 4)


## 4. Data Cleaning

Before processing the dataset, we need to handle missing values.

We first create a copy of the original movies DataFrame so that our original dataset remains unchanged.

The `overview` column contains some missing values (NaN). Since we will use movie overviews while creating tags, we replace these missing values with empty strings.

This ensures that missing overviews do not interfere with our text processing.

In [4]:
# Create a copy of the original movies DataFrame
movies_clean = movies.copy()

# Replace missing values in the overview column with empty strings
movies_clean["overview"] = movies_clean["overview"].fillna("")

### Observation

- `movies` contains the original dataset.
- `movies_clean` contains our working copy.
- Missing values in the overview column have been replaced with empty strings.

We will perform further preprocessing on `movies_clean`.

## 5. Extracting Genres

The `genres` column contains string representations of lists of dictionaries.

Each dictionary contains a genre ID and its name.

We use `ast.literal_eval()` to convert these strings into actual Python objects.

We then extract only the genre names and store them in a new column called `genres_list`.

In [5]:
# Store genres of all movies
all_genres = []

# Iterate through the genres column
for genres in movies_clean["genres"]:

    # Convert string into a Python list
    genres_list = ast.literal_eval(genres)

    # Store genres of the current movie
    movie_genres = []

    for genre in genres_list:
        movie_genres.append(genre["name"])

    all_genres.append(movie_genres)

# Add extracted genres to the DataFrame
movies_clean["genres_list"] = all_genres

### Observation

The new `genres_list` column contains the names of genres associated with each movie.

Example:

["Action", "Adventure", "Fantasy"]

We have successfully extracted genre names from the original nested structure.

## 6. Extracting Keywords

The `keywords` column also contains string representations of lists of dictionaries.

Each dictionary contains a keyword ID and its name.

Similar to genres, we use `ast.literal_eval()` to convert the strings into Python objects and extract only the keyword names.

We store the extracted keywords in a new column called `keywords_list`.

In [6]:
# Store keywords of all movies
all_keywords = []

# Iterate through the keywords column
for keywords in movies_clean["keywords"]:

    # Convert string into a Python list
    keywords_list = ast.literal_eval(keywords)

    # Store keywords of the current movie
    movie_keywords = []

    for keyword in keywords_list:
        movie_keywords.append(keyword["name"])

    all_keywords.append(movie_keywords)

# Add extracted keywords to the DataFrame
movies_clean["keywords_list"] = all_keywords

### Observation

The new `keywords_list` column contains the extracted keywords for every movie.

We now have two additional columns:

- genres_list
- keywords_list

These features will help us identify similarities between movies.

## 7. Extracting Cast

The credits dataset contains information about the cast of each movie.

The `cast` column contains string representations of lists of dictionaries. Each dictionary contains information about an actor, including their name, character and ID.

For our recommendation system, we only need the actors' names.

We use `ast.literal_eval()` to convert the string into a Python list and then extract the names of all actors.

We store the extracted cast information in a new column called `cast_list` inside `movies_clean`.

In [7]:
# Store cast names of all movies
all_cast = []

# Iterate through the cast column
for cast in credits['cast']:

    # Convert string into a Python list
    cast_list = ast.literal_eval(cast)

    # Store cast names of the current movie
    cast_names = []

    for actor in cast_list:
        cast_names.append(actor['name'])

    all_cast.append(cast_names)

# Add extracted cast names to the DataFrame
movies_clean['cast_list'] = all_cast

### Observation

The new `cast_list` column contains the names of actors associated with each movie.

Example:

["Sam Worthington", "Zoe Saldana", "Sigourney Weaver"]

We now have the cast information available for our recommendation system.

## 8. Extracting Directors

The credits dataset also contains a `crew` column.

Unlike the cast column, the crew column contains information about multiple people involved in making a movie, such as directors, producers, editors and other crew members.

We only need the directors for our recommendation system.

Each dictionary in the crew list contains a `job` key that tells us the person's role.

We iterate through the crew information and extract the names of people whose job is `"Director"`.

The extracted names are stored in a list called `directors`.

In [9]:
# Store director names of all movies
directors = []

# Iterate through the crew column
for crew in credits['crew']:

    # Convert string into a Python list
    crew_list = ast.literal_eval(crew)

    # Store director names of the current movie
    director_names = []

    for person in crew_list:

        # Check whether the person's job is Director
        if person["job"] == "Director":
            director_names.append(person['name'])

    directors.append(director_names)

# Add directors to the DataFrame
movies_clean["directors"] = directors

### Observation

The new `directors` column contains the names of directors associated with each movie.

We now have four extracted features:

- genres_list
- keywords_list
- cast_list
- directors

These features will help us identify similarities between movies.

## 9. Merging the Datasets

So far, we have worked with two separate datasets:

- `movies_clean`: Contains movie details and our extracted features.
- `credits`: Contains cast and crew information.

To build our recommendation system, we need the useful information from both datasets together.

We use Pandas' `merge()` function to combine them.

The `id` column in the movies dataset corresponds to the `movie_id` column in the credits dataset.

We use these columns as the matching keys.

After merging, we remove unnecessary duplicate columns and retain only the features required for our recommendation system.

In [21]:
# Merge movies_clean and credits using their matching IDs
movies_merged = pd.merge(
    movies_clean,
    credits,
    left_on="id",
    right_on="movie_id"
)

In [22]:
# Remove original cast, crew and duplicate title columns
movies_merged.drop(columns=["cast", "crew", "title_y"], inplace=True)

# Rename title_x to title
movies_merged.rename(columns={"title_x": "title"}, inplace=True)

In [23]:
# Select only the columns required for our recommendation system
columns_to_keep = [
    "id",
    "title",
    "overview",
    "genres_list",
    "keywords_list",
    "cast_list",
    "directors",
    "vote_average",
    "vote_count",
    "popularity",
    "release_date",
    "original_language"
]

# Keep only the selected columns
movies_merged = movies_merged[columns_to_keep].copy()

### Understanding the Merge

We used:

- `left_on="id"`: The matching column from movies_clean.
- `right_on="movie_id"`: The matching column from credits.

After merging:

- `title_x` and `title_y` were created because both datasets contained a title column.
- We removed `title_y` and renamed `title_x` to `title`.
- We removed the original `cast` and `crew` columns because we had already extracted the information we needed.

Finally, we selected only the 12 columns required for our recommendation system.

### Final Dataset After Merging

Our merged dataset contains:

- Movie ID and title
- Overview
- Genres
- Keywords
- Cast
- Directors
- Ratings and popularity
- Release date
- Original language

The dataset is now ready for further text preprocessing and tag creation.

In [24]:
print("Dataset shape:", movies_merged.shape)

print("\nColumns:")
print(movies_merged.columns.tolist())

movies_merged.head()

Dataset shape: (4803, 12)

Columns:
['id', 'title', 'overview', 'genres_list', 'keywords_list', 'cast_list', 'directors', 'vote_average', 'vote_count', 'popularity', 'release_date', 'original_language']


,id,title,overview,genres_list,keywords_list,cast_list,directors,vote_average,vote_count,popularity,release_date,original_language
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weave...",[James Cameron],7.2,11800,150.437577,2009-12-10,en
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley, ...",[Gore Verbinski],6.9,4500,139.082615,2007-05-19,en
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux, R...",[Sam Mendes],6.3,4466,107.376788,2015-10-26,en
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[Christian Bale, Michael Caine, Gary Oldman, A...",[Christopher Nolan],7.6,9106,112.312950,2012-07-16,en
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[Taylor Kitsch, Lynn Collins, Samantha Morton,...",[Andrew Stanton],6.1,2124,43.926995,2012-03-07,en


Converting list columns into strings

Why are we doing this?

Currently, genres_list, keywords_list, cast_list, and directors contain Python lists.

For our recommendation system, we want to combine all this information into text. So, we'll convert each list into a single string using " ".join().

For example:

["Action", "Adventure", "Fantasy"]

becomes:

"Action Adventure Fantasy"

In [25]:
# Convert each list into a single space-separated string

movies_merged["genres_list"] = movies_merged["genres_list"].apply(lambda x: " ".join(x))

movies_merged["keywords_list"] = movies_merged["keywords_list"].apply(lambda x: " ".join(x))

movies_merged["cast_list"] = movies_merged["cast_list"].apply(lambda x: " ".join(x))

movies_merged["directors"] = movies_merged["directors"].apply(lambda x: " ".join(x))

In [26]:
movies_merged[["title", "genres_list", "keywords_list", "cast_list", "directors"]].head()

,title,genres_list,keywords_list,cast_list,directors
0,Avatar,Action Adventure Fantasy Science Fiction,culture clash future space war space colony so...,Sam Worthington Zoe Saldana Sigourney Weaver S...,James Cameron
1,Pirates of the Caribbean: At World's End,Adventure Fantasy Action,ocean drug abuse exotic island east india trad...,Johnny Depp Orlando Bloom Keira Knightley Stel...,Gore Verbinski
2,Spectre,Action Adventure Crime,spy based on novel secret agent sequel mi6 bri...,Daniel Craig Christoph Waltz Léa Seydoux Ralph...,Sam Mendes
3,The Dark Knight Rises,Action Crime Drama Thriller,dc comics crime fighter terrorist secret ident...,Christian Bale Michael Caine Gary Oldman Anne ...,Christopher Nolan
4,John Carter,Action Adventure Science Fiction,based on novel mars medallion space travel pri...,Taylor Kitsch Lynn Collins Samantha Morton Wil...,Andrew Stanton


All four columns now contain strings instead of lists. We can combine them with the movie overview to create a single text representation for every movie.

Creating the tags column

Why are we doing this?

A content-based recommendation system recommends movies based on their content and features.

We will combine five pieces of information:
1. Movie overview
2. Genres
3. Keywords
4. Cast
5. Directors
All this information will be stored in one new column called tags.

We use apply() with axis=1 because we want to perform this operation row by row. Each row represents one movie.

In [27]:
# Combine all important movie information into one tags column

movies_merged["tags"] = movies_merged.apply(
    lambda row: row["overview"] + " " +
    row["genres_list"] + " " +
    row["keywords_list"] + " " +
    row["cast_list"] + " " +
    row["directors"],
    axis=1
)

In [28]:
movies_merged[["title", "tags"]].head()

,title,tags
0,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,Spectre,A cryptic message from Bond’s past sends him o...
3,The Dark Knight Rises,Following the death of District Attorney Harve...
4,John Carter,"John Carter is a war-weary, former military ca..."


OBSERVATION

We now have a single tags column containing all the selected information about each movie.

This column will become the foundation of our content-based recommendation system.

Converting tags to lowercase

Why are we doing this?

Computers treat uppercase and lowercase words as different text.

For example:
- Action
- action
- ACTION
These may be treated as three different words.

To avoid this, we convert all text in the tags column to lowercase.

In [29]:
# Convert all text in the tags column to lowercase

movies_merged["tags"] = movies_merged["tags"].apply(lambda x: x.lower())

In [30]:
movies_merged[["title", "tags"]].head()

,title,tags
0,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,Spectre,a cryptic message from bond’s past sends him o...
3,The Dark Knight Rises,following the death of district attorney harve...
4,John Carter,"john carter is a war-weary, former military ca..."


All movie tags are now in lowercase, giving us consistent text before further text preprocessing.

## 13. Stemming using PorterStemmer

### What is Stemming?

Stemming is an NLP (Natural Language Processing) technique that reduces words to their root or base form by removing common word endings.

For example:

- playing → play
- played → play
- plays → play
- loving → love

### Why do we need stemming?

In our movie recommendation system, words with similar meanings may appear in different forms.

For example, "playing", "played", and "plays" represent similar concepts.

Without stemming, the computer may treat these as separate words. Stemming helps reduce this difference by converting them into a common form.

We will use **PorterStemmer**, which is available through the NLTK library.

We will:
1. Create a PorterStemmer object.
2. Define a function to split the text into individual words.
3. Apply stemming to each word.
4. Join the stemmed words back into a single string.
5. Apply this function to the entire `tags` column.

In [31]:
# Create a PorterStemmer object
ps = PorterStemmer()

# Define a function to stem each word in the text
def stem_text(text):
    words = text.split()
    stemmed_words = []

    for word in words:
        stemmed_words.append(ps.stem(word))

    return " ".join(stemmed_words)

# Apply stemming to the tags column
movies_merged["tags"] = movies_merged["tags"].apply(stem_text)

In [32]:
movies_merged[["title", "tags"]].head()

,title,tags
0,Avatar,"in the 22nd century, a parapleg marin is dispa..."
1,Pirates of the Caribbean: At World's End,"captain barbossa, long believ to be dead, ha c..."
2,Spectre,a cryptic messag from bond’ past send him on a...
3,The Dark Knight Rises,follow the death of district attorney harvey d...
4,John Carter,"john carter is a war-weary, former militari ca..."


## 14. Text Vectorization using CountVectorizer

### Why do we need vectorization?

Machine learning algorithms cannot directly understand text. They work with numerical data.

Our `tags` column currently contains textual information about each movie. We need to convert this text into numerical representations so that we can compare movies mathematically.

This process is called **Text Vectorization**.

### What is CountVectorizer?

CountVectorizer is a technique provided by Scikit-learn that converts a collection of text documents into a matrix of token counts.

In our project:
- Each movie is treated as a document.
- Each unique word is treated as a feature.
- The value represents how many times that word appears in the movie's tags.

### Parameters used

- `max_features=5000`: Considers at most 5000 words, selecting the most frequent ones.
- `stop_words="english"`: Removes common English words such as "the", "is", "and", etc.

We will use `fit_transform()` to learn the vocabulary and convert our movie tags into numerical vectors.

Finally, `.toarray()` converts the resulting sparse matrix into a regular NumPy array.

In [33]:
# Create a CountVectorizer object
cv = CountVectorizer(max_features=5000, stop_words="english")

# Convert movie tags into numerical vectors
vectors = cv.fit_transform(movies_merged["tags"]).toarray()

## 15. Understanding the Generated Vectors

After applying CountVectorizer, every movie is represented by a numerical vector.

Let's understand the output.

### 1. Shape of vectors

The shape tells us:
- Number of rows = Number of movies.
- Number of columns = Number of features (words).

Since our dataset contains 4803 movies and we selected a maximum of 5000 features, the expected shape is `(4803, 5000)`.

### 2. Vocabulary

CountVectorizer creates a vocabulary containing the words it has learned from our movie tags.

We can access this vocabulary using `get_feature_names_out()`.

### 3. First movie vector

Each movie has a numerical vector representing the frequency of vocabulary words in its tags.

We can inspect the first movie's vector to understand this representation.

In [34]:
# Check the shape of the vectors
print(vectors.shape)

# Display the vocabulary
print(cv.get_feature_names_out())

# Display the first movie's vector
print(vectors[0])

(4803, 5000)
['000' '10' '11' ... 'zooey' 'zoë' 'zucker']
[0 0 0 ... 0 0 0]


## 16. Removing Purely Numeric Tokens

### Why do we need to remove numbers?

While exploring our movie tags, we may find purely numeric tokens such as years or other numbers.

These tokens may not provide useful information for identifying movie similarities.

We want to remove tokens that consist entirely of digits.

For example:
- `000` → Removed
- `1995` → Removed
- `18th` → Retained
- `19th` → Retained
- `20th` → Retained

Notice that we are only removing purely numeric tokens. Alphanumeric words should remain unchanged.

### Using Regular Expressions

We will use Python's `re` module and the `sub()` function.

The regular expression `\b\d+\b` matches complete tokens consisting only of digits.

Our function replaces these matches with an empty string.

In [35]:
# Define a function to remove purely numeric tokens
def remove_numbers(text):
    return re.sub(r'\b\d+\b', '', text)

# Apply the function to the tags column
movies_merged["tags"] = movies_merged["tags"].apply(remove_numbers)

## 17. Reapplying CountVectorizer After Cleaning

We previously applied CountVectorizer to understand how text is converted into numerical vectors.

However, we have now removed purely numeric tokens from our movie tags.

Therefore, we need to apply CountVectorizer again so that our vectors are generated using the cleaned text.

We will use the same CountVectorizer object and parameters.

The new `vectors` variable will contain the final numerical representation of our movie tags.

In [36]:
# Generate vectors again using the cleaned tags
vectors = cv.fit_transform(movies_merged["tags"]).toarray()

In [37]:
print(vectors.shape)

(4803, 5000)


## 18. Calculating Cosine Similarity

### What is Cosine Similarity?

Now that every movie has been converted into a numerical vector, we need a way to measure how similar two movies are.

For this, we will use **Cosine Similarity**.

Cosine Similarity measures the similarity between two vectors by calculating the cosine of the angle between them.

Its value generally ranges from 0 to 1 for our non-negative CountVectorizer vectors.

- 1 means the vectors point in exactly the same direction.
- 0 means the vectors are perpendicular and have no shared directional similarity.

A higher cosine similarity indicates that two movies have more similar tag representations.

### Why are we using it?

Our goal is to recommend movies that have similar content.

For example, if someone likes Avatar, we want to find other movies whose tags are similar to Avatar's tags.

Cosine Similarity allows us to compare the numerical vectors of all movies.

### Creating the Similarity Matrix

We will use `cosine_similarity()` from Scikit-learn.

It compares every movie vector with every other movie vector and generates a similarity matrix.

Since our dataset contains 4803 movies, the expected matrix shape is `(4803, 4803)`.

Each row and column represents a movie, and each cell contains the cosine similarity between the corresponding pair of movies.

In [38]:
# Calculate cosine similarity between all movie vectors
similarity = cosine_similarity(vectors)

In [39]:
print(similarity.shape)

(4803, 4803)


## 19. Building the Recommendation Function

### What are we trying to achieve?

We have already:
- Extracted important movie features.
- Combined them into a `tags` column.
- Converted the text into numerical vectors.
- Calculated cosine similarity between all movies.

Now we need to create a function that recommends movies based on a movie entered by the user.

### How will the recommendation function work?

Our function will follow these steps:

1. Accept a movie name as input.
2. Check whether the movie exists in our dataset.
3. Find the index of the selected movie.
4. Retrieve its similarity scores from the similarity matrix.
5. Sort all movies according to their similarity scores in descending order.
6. Skip the first result because it is the movie itself.
7. Display the next 10 most similar movies.

### Understanding some important concepts

- `enumerate()`: Attaches an index to each similarity score.
- `sorted()`: Sorts the movie-index and similarity-score pairs.
- `reverse=True`: Sorts in descending order, placing the highest similarity first.
- `key=lambda x: x[1]`: Sorts using the similarity score rather than the movie index.
- `[1:11]`: Selects 10 recommendations while skipping the first result (the movie itself).
- `iloc[]`: Retrieves a row using its numerical index.

We will create a function called `recommend()` that performs all these steps automatically.

In [40]:
def recommend(movie):
    if movie not in movies_merged["title"].values:
        print("Movie not found in our dataset!")
        return

    movie_index = movies_merged[movies_merged["title"] == movie].index[0]

    distances = similarity[movie_index]

    movie_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:11]

    for i in movie_list:
        print(movies_merged.iloc[i[0]]["title"])

In [41]:
recommend("Avatar")

Aliens
Star Trek Into Darkness
Armageddon
The Dark Knight
Aliens vs Predator: Requiem
Moonraker
Lifeforce
Lockout
Star Trek
Independence Day


In [43]:
recommend("ramayana")

Movie not found in our dataset!


## 20. Saving and Loading the Model using Pickle

### Why do we need to save our model?

Currently, our recommendation system requires us to execute all previous steps:

- Loading the datasets.
- Cleaning and preprocessing the data.
- Creating tags.
- Applying CountVectorizer.
- Calculating cosine similarity.

These operations can take time, especially when the dataset becomes larger.

We don't want to repeat all these operations every time we want to use our recommendation system.

Instead, we can save the processed movie dataset and similarity matrix so that we can load and reuse them later.

### What is Pickle?

Pickle is a Python module used to serialize and deserialize Python objects.

- Serialization: Converting a Python object into a format that can be saved to a file.
- Deserialization: Loading the saved file and reconstructing the original Python object.

We will use two functions:

- `pickle.dump()`: Saves a Python object.
- `pickle.load()`: Loads a previously saved Python object.

### File modes

- `wb`: Write Binary — used when saving objects.
- `rb`: Read Binary — used when loading objects.

We will save two objects:

1. `movies_merged` as `movies.pkl`
2. `similarity` as `similarity.pkl`

These two files will allow us to reuse our processed movie data and similarity matrix without calculating everything again.

In [50]:
import pickle

pickle.dump(movies_merged, open("../models/movies.pkl", "wb"))
pickle.dump(similarity, open("../models/similarity.pkl", "wb"))

# wb = write binary

### Loading the Saved Objects

Now that we have saved our objects, we can load them back into Python.

We will store them in:
- `movies` — contains the processed movie dataset.
- `similarity_matrix` — contains the cosine similarity matrix.

Notice that we are using `rb` mode because we are reading the saved binary files.

In [51]:
movies = pickle.load(open("../models/movies.pkl", "rb"))
similarity_matrix = pickle.load(open("../models/similarity.pkl", "rb"))

# rb = read binary

In [52]:
print(movies.shape)
print(similarity_matrix.shape)

(4803, 13)
(4803, 4803)


In [53]:
print(movies["title"].head())
print(similarity_matrix[0][:5])

0                                      Avatar
1    Pirates of the Caribbean: At World's End
2                                     Spectre
3                       The Dark Knight Rises
4                                 John Carter
Name: title, dtype: str
[1.         0.10881351 0.07927124 0.22212349 0.22284664]


### Creating a Recommendation Function Using Saved Objects

We have successfully loaded our saved objects.

Now, we can create another recommendation function called `recommend_saved()`.

Unlike our earlier `recommend()` function, this function uses the loaded objects:

- `movies` instead of `movies_merged`
- `similarity_matrix` instead of `similarity`

This allows us to generate recommendations using the saved data without repeating the entire preprocessing and similarity calculation process.

In [54]:
def recommend_saved(movie):
    if movie not in movies["title"].values:
        print("Movie not found in our dataset!")
        return

    movie_index = movies[movies["title"] == movie].index[0]
    distances = similarity_matrix[movie_index]

    movie_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:11]

    for i in movie_list:
        print(movies.iloc[i[0]]["title"])

In [55]:
recommend_saved("Avatar")

Aliens
Star Trek Into Darkness
Armageddon
The Dark Knight
Aliens vs Predator: Requiem
Moonraker
Lifeforce
Lockout
Star Trek
Independence Day
